In [1]:
import numpy as np
import statsmodels.api as sm
from sklearn.decomposition import PCA
from concurrent.futures import ProcessPoolExecutor, as_completed
import itertools
import logging
from one.api import ONE
from brainbox.io.one import SessionLoader
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from collections import defaultdict
import pandas as pd
from manifold.decoding.functions.utils import check_config_decoding
import numpy as np
import pickle as pkl
from manifold.decoding.functions import nulldistributions
from communication_subspace.ibl_communication.utils import load_widefield_epoch
from tqdm import tqdm
from iblatlas.atlas import AllenAtlas
from iblatlas.regions import BrainRegions
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
import warnings
from manifold.utils import get_trial_masks
from scipy.stats import pointbiserialr
from manifold.widefield_ppi import beryl_mapping,aggregate_by_parent
from matplotlib import pyplot as plt
from glob import glob
import os

In [2]:
with open('../data/generated/prior_sig/CSK-im-009/f7d46a15-9498-40dc-90da-fb977ce844be/MOs_both_hemispheres_pseudo_ids_-1_200.pkl','rb') as f:
    data = pkl.load(f)

In [23]:
filenames = glob('../data/generated/prior_sig/**/*_both*.pkl', recursive=True)
filenames

['../data/generated/prior_sig/CSK-im-009/f7d46a15-9498-40dc-90da-fb977ce844be/VISp_both_hemispheres_pseudo_ids_-1_200.pkl',
 '../data/generated/prior_sig/CSK-im-009/f7d46a15-9498-40dc-90da-fb977ce844be/MOs_both_hemispheres_pseudo_ids_-1_200.pkl']

In [26]:
data['fit'][45]['run_id']

0

In [27]:
indexers = ["subject", "eid", "region", "N_units"]

failed_load = 0
datalist = []
for fname in filenames:
    with open(fname,'rb') as f:
        data = pkl.load(f)
    if data['fit'] is None:
        continue
    for iteration in range(len(data['fit'])):
        tmpdict = {**{x: data[x] for x in indexers},
                    "fold": -1,
                    "pseudo_id": data["fit"][iteration]["pseudo_id"],
                    "run_id": data["fit"][iteration]["run_id"] + 1,
                    "score_test": data["fit"][iteration]["scores_test_full"],
                    "n_trials": sum(data['fit'][iteration]['mask'][0]),
            }
        datalist.append(tmpdict)

In [28]:
resultsdf = pd.DataFrame(datalist)

,subject,eid,region,N_units,fold,pseudo_id,run_id,score_test,n_trials
0,CSK-im-009,f7d46a15-9498-40dc-90da-fb977ce844be,[VISp],243,-1,-1,1,-0.003261,767
1,CSK-im-009,f7d46a15-9498-40dc-90da-fb977ce844be,[VISp],243,-1,1,1,-0.014581,767
2,CSK-im-009,f7d46a15-9498-40dc-90da-fb977ce844be,[VISp],243,-1,2,1,-0.005149,767
3,CSK-im-009,f7d46a15-9498-40dc-90da-fb977ce844be,[VISp],243,-1,3,1,-0.009142,767
4,CSK-im-009,f7d46a15-9498-40dc-90da-fb977ce844be,[VISp],243,-1,4,1,-0.015135,767
...,...,...,...,...,...,...,...,...,...
397,CSK-im-009,f7d46a15-9498-40dc-90da-fb977ce844be,[VISp],243,-1,196,1,-0.003295,767
398,CSK-im-009,f7d46a15-9498-40dc-90da-fb977ce844be,[VISp],243,-1,197,1,-0.021585,767
399,CSK-im-009,f7d46a15-9498-40dc-90da-fb977ce844be,[VISp],243,-1,198,1,-0.016308,767
400,CSK-im-009,f7d46a15-9498-40dc-90da-fb977ce844be,[VISp],243,-1,199,1,-0.014291,767


In [30]:
data['fit'][0]['n_folds']

5